# Updated Lung1 Extraction Notebook

Clean Lung1 audit, RTSTRUCT mask handling, and PyRadiomics extraction using the same fixed YAML settings.

**Important before running:** update the path configuration cell for your local machine. Keep `params_clean_updated.yaml` in the same working directory as this notebook, or edit the `RADIOMICS_YAML` path.

**Methodology notes:** CV remains the primary stability-screening metric. Susceptibility Index (SI) is supplementary. Any LDA step is fitted only inside training folds/splits to avoid leakage.


In [38]:
# Updated Lung1 Audit and Radiomics Extraction Notebook
# Cleaned to remove redundant stable-feature cells.
#
# Purpose:
# - Audit Lung1 CT/RTSTRUCT metadata.
# - Build the 2-year OS label table.
# - Extract Lung1 radiomics using the same PyRadiomics YAML settings as the phantom phase.
#
# Note:
# - Feature selection, SI, LDA, and ML evaluation are handled in updated_ml_lda.py.

## 0. Shared imports

In [39]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# In Jupyter, display() is available automatically.
# If you run this as a .py script outside Jupyter, replace display(x) with print(x).

## 1. Lung1 local DICOM audit

In [40]:
# ==========================================
# Lung1 local DICOM audit from manifest folder
# LOCAL DESKTOP VERSION + ORGANIZED OUTPUTS
# ==========================================

from pathlib import Path
from time import perf_counter
import pandas as pd
import numpy as np

import pydicom
from pydicom.errors import InvalidDicomError

from tqdm.auto import tqdm


# ==========================================
# PROJECT AND PATH CONFIGURATION
# ==========================================

PROJECT_ROOT = Path(r"C:\Users\Ideanuodo\Desktop\Updated Code")

# Folder that contains LUNG1-001, LUNG1-002, ...
LUNG1_ROOT = Path(r"C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\manifest-1603198545583\NSCLC-Radiomics")

# Main organized Lung1 output folder
OUT_LUNG1 = PROJECT_ROOT / "outputs_lung1_clean"
OUT_LUNG1.mkdir(parents=True, exist_ok=True)

# Organized subfolders
OUT_AUDIT = OUT_LUNG1 / "01_dicom_audit"
OUT_PREML = OUT_LUNG1 / "02_preML_tables"
OUT_RADIOMICS = OUT_LUNG1 / "03_radiomics_extraction"
OUT_QC = OUT_LUNG1 / "04_qc_logs"
OUT_MANIFEST = OUT_LUNG1 / "05_manifest"

for folder in [
    OUT_AUDIT,
    OUT_PREML,
    OUT_RADIOMICS,
    OUT_QC,
    OUT_MANIFEST,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Common audit output paths
PATIENT_SUMMARY_CSV = OUT_AUDIT / "lung1_patient_modality_summary.csv"
CT_AUDIT_CSV = OUT_AUDIT / "lung1_ct_series_audit.csv"
ALL_SERIES_CSV = OUT_AUDIT / "lung1_all_series_inventory.csv"

# Set this to a small number for a dry run, e.g. 10.
# Set to None for all patients.
PATIENT_LIMIT = None


# ==========================================
# PATH CHECK
# ==========================================

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LUNG1_ROOT:", LUNG1_ROOT)
print("OUT_LUNG1:", OUT_LUNG1)
print("LUNG1_ROOT exists:", LUNG1_ROOT.exists())
print("Using OneDrive path:", "OneDrive" in str(LUNG1_ROOT))

if not LUNG1_ROOT.exists():
    raise FileNotFoundError(
        f"LUNG1_ROOT does not exist. Check your folder structure:\n{LUNG1_ROOT}"
    )


# ==========================================
# HELPER FUNCTIONS
# ==========================================

def format_seconds(seconds: float) -> str:
    """Convert seconds into readable h/m/s format."""
    seconds = int(max(seconds, 0))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"


def safe_get(ds, name, default=np.nan):
    """Safely get a DICOM attribute by keyword."""
    return getattr(ds, name, default)


def read_dicom_header(fp: Path):
    """
    Read only the DICOM header, not the pixel data.
    This is much faster for audit purposes.
    """
    try:
        return pydicom.dcmread(
            str(fp),
            stop_before_pixels=True,
            force=True
        )
    except (InvalidDicomError, Exception):
        return None


def find_dicom_files(folder: Path):
    """
    TCIA downloads sometimes use .dcm, .DCM, or no extension.
    This function first tries DICOM extensions, then falls back to all files.
    """
    files = list(folder.rglob("*.dcm")) + list(folder.rglob("*.DCM"))

    if len(files) == 0:
        files = [p for p in folder.rglob("*") if p.is_file()]

    return files


def audit_one_patient(patient_dir: Path):
    """
    Audits one Lung1 patient folder.

    Returns:
        patient_summary: summary dictionary for the patient
        ct_rows: list of CT acquisition metadata rows
        modality_df: dataframe of all detected DICOM series
    """
    files = find_dicom_files(patient_dir)

    series_seen = {}
    modality_series_rows = []

    for fp in files:
        ds = read_dicom_header(fp)

        if ds is None:
            continue

        mod = safe_get(ds, "Modality", "")
        series_uid = safe_get(ds, "SeriesInstanceUID", "")
        study_uid = safe_get(ds, "StudyInstanceUID", "")
        sop_uid = safe_get(ds, "SOPInstanceUID", "")
        patient_id = safe_get(ds, "PatientID", patient_dir.name)

        if not series_uid:
            continue

        # Keep one representative file per DICOM series.
        if series_uid not in series_seen:
            series_seen[series_uid] = {
                "PatientID": str(patient_id),
                "PatientFolder": str(patient_dir),
                "Modality": str(mod),
                "StudyInstanceUID": str(study_uid),
                "SeriesInstanceUID": str(series_uid),
                "ExampleFile": str(fp),
                "SOPInstanceUID": str(sop_uid),
            }

    for s in series_seen.values():
        modality_series_rows.append(s)

    modality_df = pd.DataFrame(modality_series_rows)

    if modality_df.empty:
        patient_summary = {
            "PatientID": patient_dir.name,
            "PatientFolder": str(patient_dir),
            "n_series_total": 0,
            "has_CT": False,
            "has_SEG": False,
            "has_RTSTRUCT": False,
            "modalities_present": "",
        }

        return patient_summary, [], modality_df

    mods = set(modality_df["Modality"].astype(str).tolist())
    pid = modality_df["PatientID"].astype(str).iloc[0]

    patient_summary = {
        "PatientID": pid,
        "PatientFolder": str(patient_dir),
        "n_series_total": int(modality_df.shape[0]),
        "has_CT": "CT" in mods,
        "has_SEG": "SEG" in mods,
        "has_RTSTRUCT": "RTSTRUCT" in mods,
        "modalities_present": ",".join(sorted([m for m in mods if m])),
    }

    # Extract CT acquisition metadata from representative CT headers.
    ct_rows = []
    ct_series = modality_df[modality_df["Modality"] == "CT"].copy()

    for _, r in ct_series.iterrows():
        ex = Path(r["ExampleFile"])
        ds = read_dicom_header(ex)

        if ds is None:
            continue

        pix = safe_get(ds, "PixelSpacing", None)

        try:
            pix0 = float(pix[0])
            pix1 = float(pix[1])
        except Exception:
            pix0, pix1 = (np.nan, np.nan)

        ct_rows.append({
            "PatientID": str(pid),
            "SeriesInstanceUID": str(safe_get(ds, "SeriesInstanceUID", "")),
            "StudyInstanceUID": str(safe_get(ds, "StudyInstanceUID", "")),
            "ExampleFile": str(ex),

            "Manufacturer": str(safe_get(ds, "Manufacturer", "")),
            "ManufacturerModelName": str(safe_get(ds, "ManufacturerModelName", "")),
            "StationName": str(safe_get(ds, "StationName", "")),

            "ConvolutionKernel": str(safe_get(ds, "ConvolutionKernel", "")),
            "ReconstructionAlgorithm": str(safe_get(ds, "ReconstructionAlgorithm", "")),

            "SliceThickness": safe_get(ds, "SliceThickness", np.nan),
            "SpacingBetweenSlices": safe_get(ds, "SpacingBetweenSlices", np.nan),

            "PixelSpacingRow": pix0,
            "PixelSpacingCol": pix1,

            "KVP": safe_get(ds, "KVP", np.nan),
            "TubeCurrent_mA": safe_get(
                ds,
                "XRayTubeCurrent",
                safe_get(ds, "TubeCurrent", np.nan)
            ),
            "Exposure_mAs": safe_get(ds, "Exposure", np.nan),
            "ExposureTime_ms": safe_get(ds, "ExposureTime", np.nan),
            "CTDIvol_mGy": safe_get(ds, "CTDIvol", np.nan),

            "Rows": safe_get(ds, "Rows", np.nan),
            "Columns": safe_get(ds, "Columns", np.nan),

            "StudyDate": str(safe_get(ds, "StudyDate", "")),
            "SeriesDate": str(safe_get(ds, "SeriesDate", "")),
        })

    return patient_summary, ct_rows, modality_df




PROJECT_ROOT: C:\Users\Ideanuodo\Desktop\Updated Code
LUNG1_ROOT: C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\manifest-1603198545583\NSCLC-Radiomics
OUT_LUNG1: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean
LUNG1_ROOT exists: True
Using OneDrive path: False


In [41]:
# ==========================================
# RUN AUDIT WITH PROGRESS BAR
# ==========================================

patient_dirs = sorted([
    p for p in LUNG1_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("LUNG1-")
])

if PATIENT_LIMIT is not None:
    patient_dirs = patient_dirs[:PATIENT_LIMIT]

print("Patients to audit:", len(patient_dirs))

all_patient_summaries = []
all_ct_rows = []
all_modality_series = []

start_time = perf_counter()

progress_bar = tqdm(
    patient_dirs,
    total=len(patient_dirs),
    desc="Auditing Lung1 DICOM folders",
    unit="patient",
    dynamic_ncols=True,
)

for i, pdir in enumerate(progress_bar, start=1):
    psum, ct_rows, mod_df = audit_one_patient(pdir)

    all_patient_summaries.append(psum)
    all_ct_rows.extend(ct_rows)

    if not mod_df.empty:
        all_modality_series.append(mod_df)

    elapsed = perf_counter() - start_time
    avg_time = elapsed / i
    remaining = len(patient_dirs) - i
    eta = avg_time * remaining

    progress_bar.set_postfix({
        "current": pdir.name,
        "elapsed": format_seconds(elapsed),
        "ETA": format_seconds(eta),
    })


# ==========================================
# ASSEMBLE AUDIT TABLES
# ==========================================

patients_df = pd.DataFrame(all_patient_summaries)
ct_df = pd.DataFrame(all_ct_rows)

mods_df = (
    pd.concat(all_modality_series, ignore_index=True)
    if len(all_modality_series)
    else pd.DataFrame()
)

display(patients_df.head())
display(ct_df.head())
display(mods_df.head())


# ==========================================
# AUDIT SUMMARY
# ==========================================

print("\nPatients total:", patients_df.shape[0])

if not patients_df.empty:
    print("Patients with CT:", int(patients_df["has_CT"].sum()))
    print("Patients with SEG:", int(patients_df["has_SEG"].sum()))
    print("Patients with RTSTRUCT:", int(patients_df["has_RTSTRUCT"].sum()))

print("\nCT series count:", ct_df.shape[0])

if not ct_df.empty:
    print("\nManufacturer distribution:")
    display(ct_df["Manufacturer"].value_counts(dropna=False))

    print("\nKernel distribution (ConvolutionKernel):")
    display(
        ct_df["ConvolutionKernel"]
        .replace("", np.nan)
        .value_counts(dropna=False)
        .head(25)
    )

    print("\nSlice thickness distribution:")
    display(pd.to_numeric(ct_df["SliceThickness"], errors="coerce").describe())

    print("\nPixel spacing examples (row,col):")
    pix_pairs = (
        ct_df[["PixelSpacingRow", "PixelSpacingCol"]]
        .dropna()
        .round(4)
        .astype(str)
        .agg(",".join, axis=1)
        .value_counts()
        .head(20)
    )
    display(pix_pairs)

    multi_ct = ct_df.groupby("PatientID")["SeriesInstanceUID"].nunique()

    print("\nPatients with >1 CT series:", int((multi_ct > 1).sum()))

    if int((multi_ct > 1).sum()) > 0:
        display(
            multi_ct[multi_ct > 1]
            .sort_values(ascending=False)
            .head(20)
        )


# ==========================================
# SAVE AUDIT OUTPUTS
# ==========================================

patients_df.to_csv(PATIENT_SUMMARY_CSV, index=False)
ct_df.to_csv(CT_AUDIT_CSV, index=False)
mods_df.to_csv(ALL_SERIES_CSV, index=False)

print("\nSaved audit outputs:")
print(PATIENT_SUMMARY_CSV.resolve())
print(CT_AUDIT_CSV.resolve())
print(ALL_SERIES_CSV.resolve())

print("\nUsing OneDrive path:", "OneDrive" in str(LUNG1_ROOT.resolve()))

Patients to audit: 422


Auditing Lung1 DICOM folders: 100%|██████████| 422/422 [11:15<00:00,  1.60s/patient, current=LUNG1-422, elapsed=11m 15s, ETA=0s]    


,PatientID,PatientFolder,n_series_total,has_CT,has_SEG,has_RTSTRUCT,modalities_present
0,LUNG1-001,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,3,True,True,True,"CT,RTSTRUCT,SEG"
1,LUNG1-002,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,3,True,True,True,"CT,RTSTRUCT,SEG"
2,LUNG1-003,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,3,True,True,True,"CT,RTSTRUCT,SEG"
3,LUNG1-004,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,3,True,True,True,"CT,RTSTRUCT,SEG"
4,LUNG1-005,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,3,True,True,True,"CT,RTSTRUCT,SEG"


,PatientID,SeriesInstanceUID,StudyInstanceUID,ExampleFile,Manufacturer,ManufacturerModelName,StationName,ConvolutionKernel,ReconstructionAlgorithm,SliceThickness,...,PixelSpacingCol,KVP,TubeCurrent_mA,Exposure_mAs,ExposureTime_ms,CTDIvol_mGy,Rows,Columns,StudyDate,SeriesDate
0,LUNG1-001,1.3.6.1.4.1.32722.99.99.2989917765213423750108...,1.3.6.1.4.1.32722.99.99.2393413539117143687725...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,SIEMENS,Biograph 40,,B19f,,3.0,...,0.976562,120.0,80.0,400.0,361.0,NaN,512,512,20080918,20080918
1,LUNG1-002,1.3.6.1.4.1.32722.99.99.2329880015517990803358...,1.3.6.1.4.1.32722.99.99.2037150038059966416957...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,"CMS, Inc.",XiO,,,,3.0,...,0.977000,NaN,NaN,NaN,NaN,NaN,512,512,20140101,
2,LUNG1-003,1.3.6.1.4.1.32722.99.99.2389222799296192439904...,1.3.6.1.4.1.32722.99.99.2477262867958601216867...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,"CMS, Inc.",XiO,,,,3.0,...,0.977000,NaN,NaN,NaN,NaN,NaN,512,512,20140101,
3,LUNG1-004,1.3.6.1.4.1.32722.99.99.2809816144625926346520...,1.3.6.1.4.1.32722.99.99.2026036697036260886779...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,SIEMENS,Sensation Open,,B31s,,3.0,...,0.976562,120.0,80.0,66.0,1000.0,NaN,512,512,20060924,20060924
4,LUNG1-005,1.3.6.1.4.1.32722.99.99.3490584753983772067630...,1.3.6.1.4.1.32722.99.99.7196186628043392557101...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,"CMS, Inc.",XiO,,,,3.0,...,0.977000,NaN,NaN,NaN,NaN,NaN,512,512,20140101,


,PatientID,PatientFolder,Modality,StudyInstanceUID,SeriesInstanceUID,ExampleFile,SOPInstanceUID
0,LUNG1-001,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,CT,1.3.6.1.4.1.32722.99.99.2393413539117143687725...,1.3.6.1.4.1.32722.99.99.2989917765213423750108...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,1.3.6.1.4.1.32722.99.99.3078801584366390810576...
1,LUNG1-001,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,RTSTRUCT,1.3.6.1.4.1.32722.99.99.2393413539117143687725...,1.3.6.1.4.1.32722.99.99.2279381215866080725084...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,1.3.6.1.4.1.32722.99.99.6468474582136099606367...
2,LUNG1-001,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,SEG,1.3.6.1.4.1.32722.99.99.2393413539117143687725...,1.2.276.0.7230010.3.1.3.2323910823.20524.15972...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,1.2.276.0.7230010.3.1.4.2323910823.20524.15972...
3,LUNG1-002,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,CT,1.3.6.1.4.1.32722.99.99.2037150038059966416957...,1.3.6.1.4.1.32722.99.99.2329880015517990803358...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,1.3.6.1.4.1.32722.99.99.1466905188969397179902...
4,LUNG1-002,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,SEG,1.3.6.1.4.1.32722.99.99.2037150038059966416957...,1.2.276.0.7230010.3.1.3.2323910823.11504.15972...,C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\m...,1.2.276.0.7230010.3.1.4.2323910823.11504.15972...


...


Slice thickness distribution:


count    422.0
mean       3.0
std        0.0
min        3.0
25%        3.0
50%        3.0
75%        3.0
max        3.0
Name: SliceThickness, dtype: float64


Pixel spacing examples (row,col):


0.9766,0.9766    325
0.977,0.977       93
0.898,0.898        1
0.812,0.812        1
0.7207,0.7207      1
0.9121,0.9121      1
Name: count, dtype: int64


Patients with >1 CT series: 0

Saved audit outputs:
C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_patient_modality_summary.csv
C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_ct_series_audit.csv
C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_all_series_inventory.csv

Using OneDrive path: False


## 2. Lung1 pre-ML report and clinical + CT join

In [42]:
# ==========================================
# 2. Lung1 pre-ML report and clinical + CT join
# Cell 1: Local path setup
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Folder where your updated notebooks/outputs are stored
PROJECT_ROOT = Path(r"C:\Users\Ideanuodo\Desktop\Updated Code")

# Folder that contains LUNG1-001, LUNG1-002, ...
# This is the actual local dataset location.
LUNG1_ROOT = Path(
    r"C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\manifest-1603198545583\NSCLC-Radiomics"
)

# Organized Lung1 output folder remains inside Updated Code
OUT_LUNG1 = PROJECT_ROOT / "outputs_lung1_clean"

OUT_AUDIT = OUT_LUNG1 / "01_dicom_audit"
OUT_PREML = OUT_LUNG1 / "02_preML_tables"
OUT_RADIOMICS = OUT_LUNG1 / "03_radiomics_extraction"
OUT_QC = OUT_LUNG1 / "04_qc_logs"
OUT_MANIFEST = OUT_LUNG1 / "05_manifest"
OUT_FIGURES = OUT_LUNG1 / "figures"

for folder in [
    OUT_AUDIT,
    OUT_PREML,
    OUT_RADIOMICS,
    OUT_QC,
    OUT_MANIFEST,
    OUT_FIGURES,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Audit outputs from the previous DICOM audit section
PATIENTS_CSV = OUT_AUDIT / "lung1_patient_modality_summary.csv"
CT_CSV = OUT_AUDIT / "lung1_ct_series_audit.csv"
SERIES_CSV = OUT_AUDIT / "lung1_all_series_inventory.csv"

# Pre-ML outputs
LUNG1_JOIN_CSV = OUT_PREML / "lung1_join_clinical_ct_OS2y.csv"
SURVIVAL_HIST_PNG = OUT_FIGURES / "lung1_survival_time_distribution.png"

# Clinical CSV filename
CLIN_FILENAME = "NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019 (1).csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LUNG1_ROOT:", LUNG1_ROOT)
print("OUT_AUDIT:", OUT_AUDIT)
print("OUT_PREML:", OUT_PREML)
print("OUT_RADIOMICS:", OUT_RADIOMICS)
print("OUT_QC:", OUT_QC)
print("OUT_FIGURES:", OUT_FIGURES)

PROJECT_ROOT: C:\Users\Ideanuodo\Desktop\Updated Code
LUNG1_ROOT: C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\manifest-1603198545583\NSCLC-Radiomics
OUT_AUDIT: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit
OUT_PREML: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\02_preML_tables
OUT_RADIOMICS: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\03_radiomics_extraction
OUT_QC: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\04_qc_logs
OUT_FIGURES: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\figures


In [43]:
# ==========================================
# Cell 2: Path checks
# ==========================================

print("LUNG1_ROOT exists:", LUNG1_ROOT.exists())
print("PATIENTS_CSV exists:", PATIENTS_CSV.exists())
print("CT_CSV exists:", CT_CSV.exists())
print("SERIES_CSV exists:", SERIES_CSV.exists())

print("\nUsing OneDrive path?")
print("PROJECT_ROOT:", "OneDrive" in str(PROJECT_ROOT))
print("LUNG1_ROOT:", "OneDrive" in str(LUNG1_ROOT))
print("OUT_LUNG1:", "OneDrive" in str(OUT_LUNG1))

if not LUNG1_ROOT.exists():
    raise FileNotFoundError(f"LUNG1_ROOT does not exist:\n{LUNG1_ROOT}")

for required_csv in [PATIENTS_CSV, CT_CSV, SERIES_CSV]:
    if not required_csv.exists():
        raise FileNotFoundError(
            f"Required audit output not found:\n{required_csv}\n"
            "Run the Lung1 DICOM audit section first."
        )
        
# ==========================================
# Cell 3: Helper function for robust local file search
# ==========================================

def find_file(filename: str, search_roots):
    """
    Search for a filename under multiple local roots.
    Returns the first matching Path or None.
    """
    for root in search_roots:
        root = Path(root)

        if root.is_file() and root.name == filename:
            return root

        if root.is_dir():
            direct = root / filename

            if direct.exists():
                return direct

            hits = list(root.rglob(filename))

            if hits:
                return hits[0]

    return None

LUNG1_ROOT exists: True
PATIENTS_CSV exists: True
CT_CSV exists: True
SERIES_CSV exists: True

Using OneDrive path?
PROJECT_ROOT: False
LUNG1_ROOT: False
OUT_LUNG1: False


In [44]:
# ==========================================
# Search for possible Lung1 clinical CSV files
# ==========================================

from pathlib import Path

search_roots = [
    PROJECT_ROOT,
    OUT_LUNG1,
    LUNG1_ROOT,
    LUNG1_ROOT.parent,
    LUNG1_ROOT.parent.parent,
    Path(r"C:\Users\Ideanuodo\Desktop\THESIS"),
    Path(r"C:\Users\Ideanuodo\Desktop"),
    Path.cwd(),
]

possible_clinical_files = []

for root in search_roots:
    root = Path(root)

    if not root.exists():
        continue

    for fp in root.rglob("*.csv"):
        name_lower = fp.name.lower()

        if (
            "clinical" in name_lower
            or "lung1" in name_lower
            or "nsclc" in name_lower
            or "survival" in name_lower
        ):
            possible_clinical_files.append(fp)

# Remove duplicates while preserving order
seen = set()
unique_files = []

for fp in possible_clinical_files:
    key = str(fp).lower()
    if key not in seen:
        unique_files.append(fp)
        seen.add(key)

print("Possible clinical CSV files found:", len(unique_files))

for i, fp in enumerate(unique_files, start=1):
    print(f"{i}. {fp}")

Possible clinical CSV files found: 9
1. C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_all_series_inventory.csv
2. C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_ct_series_audit.csv
3. C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_patient_modality_summary.csv
4. C:\Users\Ideanuodo\Desktop\THESIS\lung1_audit_outputs\lung1_all_series_inventory.csv
5. C:\Users\Ideanuodo\Desktop\THESIS\lung1_audit_outputs\lung1_ct_series_audit.csv
6. C:\Users\Ideanuodo\Desktop\THESIS\lung1_audit_outputs\lung1_patient_modality_summary.csv
7. C:\Users\Ideanuodo\Desktop\THESIS\lung1_preML_outputs\lung1_join_clinical_ct_OS2y.csv
8. C:\Users\Ideanuodo\Desktop\THESIS\lung1_radiomics_outputs\lung1_radiomics_features.csv
9. C:\Users\Ideanuodo\Desktop\THESIS\lung1_radiomics_outputs\lung1_radiomics_qc.csv


In [46]:
# ==========================================
# Restore Lung1 clinical + CT joined table
# ==========================================

from pathlib import Path
import shutil
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Ideanuodo\Desktop\Updated Code")

OUT_LUNG1 = PROJECT_ROOT / "outputs_lung1_clean"
OUT_PREML = OUT_LUNG1 / "02_preML_tables"
OUT_PREML.mkdir(parents=True, exist_ok=True)

old_join_path = Path(
    r"C:\Users\Ideanuodo\Desktop\THESIS\lung1_preML_outputs\lung1_join_clinical_ct_OS2y.csv"
)

LUNG1_JOIN_CSV = OUT_PREML / "lung1_join_clinical_ct_OS2y.csv"

print("Old joined table exists:", old_join_path.exists())
print("New joined table exists before copy:", LUNG1_JOIN_CSV.exists())

if not LUNG1_JOIN_CSV.exists():
    if not old_join_path.exists():
        raise FileNotFoundError(f"Old joined table not found:\n{old_join_path}")
    shutil.copy2(old_join_path, LUNG1_JOIN_CSV)
    print("Copied joined table to organized folder.")

lung1_join = pd.read_csv(LUNG1_JOIN_CSV)

print("\nLoaded joined table:")
print(LUNG1_JOIN_CSV)
print("Shape:", lung1_join.shape)

required_cols = [
    "PatientID",
    "OS_2y",
    "Manufacturer",
    "ManufacturerModelName",
    "ConvolutionKernel",
    "SliceThickness",
    "PixelSpacingRow",
    "PixelSpacingCol",
    "KVP",
]

missing = [c for c in required_cols if c not in lung1_join.columns]
print("Missing columns:", missing)

if missing:
    raise KeyError(f"Missing required columns: {missing}")

display(lung1_join.head())

Old joined table exists: True
New joined table exists before copy: False
Copied joined table to organized folder.

Loaded joined table:
C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\02_preML_tables\lung1_join_clinical_ct_OS2y.csv
Shape: (420, 18)
Missing columns: []


,PatientID,age,clinical.T.Stage,Clinical.N.Stage,Clinical.M.Stage,Overall.Stage,Histology,gender,Survival.time,deadstatus.event,OS_2y,Manufacturer,ManufacturerModelName,ConvolutionKernel,SliceThickness,PixelSpacingRow,PixelSpacingCol,KVP
0,LUNG1-001,78.7515,2.0,3,0,IIIb,large cell,male,2165,1,0.0,SIEMENS,Biograph 40,B19f,3.0,0.976562,0.976562,120.0
1,LUNG1-002,83.8001,2.0,0,0,I,squamous cell carcinoma,male,155,1,1.0,"CMS, Inc.",XiO,NaN,3.0,0.977000,0.977000,NaN
2,LUNG1-003,68.1807,2.0,3,0,IIIb,large cell,male,256,1,1.0,"CMS, Inc.",XiO,NaN,3.0,0.977000,0.977000,NaN
3,LUNG1-004,70.8802,2.0,1,0,II,squamous cell carcinoma,male,141,1,1.0,SIEMENS,Sensation Open,B31s,3.0,0.976562,0.976562,120.0
4,LUNG1-005,80.4819,4.0,2,0,IIIb,squamous cell carcinoma,male,353,1,1.0,"CMS, Inc.",XiO,NaN,3.0,0.977000,0.977000,NaN


In [47]:
# ==========================================
# Check if joined table is usable
# ==========================================

required_cols = [
    "PatientID",
    "OS_2y",
    "Manufacturer",
    "ManufacturerModelName",
    "ConvolutionKernel",
    "SliceThickness",
    "PixelSpacingRow",
    "PixelSpacingCol",
    "KVP",
]

missing = [c for c in required_cols if c not in lung1_join.columns]

print("Missing columns:", missing)

if len(missing) == 0:
    print("Joined table is usable. Continue to Lung1 radiomics extraction.")
else:
    raise KeyError(f"Missing required columns: {missing}")


Missing columns: []
Joined table is usable. Continue to Lung1 radiomics extraction.


## 3. Lung1 radiomics extraction using same YAML as phantom

In [48]:
# ==========================================
# 3. Lung1 radiomics extraction
# Cell 1: Path setup
# ==========================================

from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

import pydicom
from pydicom.errors import InvalidDicomError

import SimpleITK as sitk
from radiomics import featureextractor

from tqdm.auto import tqdm

try:
    from rt_utils import RTStructBuilder
except Exception as e:
    raise ImportError(
        "rt-utils is required. Install it with: pip install rt-utils"
    ) from e


PROJECT_ROOT = Path(r"C:\Users\Ideanuodo\Desktop\Updated Code")

# Actual dataset location
LUNG1_ROOT = Path(
    r"C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\manifest-1603198545583\NSCLC-Radiomics"
)

# Organized output folders
OUT_LUNG1 = PROJECT_ROOT / "outputs_lung1_clean"

OUT_AUDIT = OUT_LUNG1 / "01_dicom_audit"
OUT_PREML = OUT_LUNG1 / "02_preML_tables"
OUT_RADIOMICS = OUT_LUNG1 / "03_radiomics_extraction"
OUT_QC = OUT_LUNG1 / "04_qc_logs"

for folder in [OUT_AUDIT, OUT_PREML, OUT_RADIOMICS, OUT_QC]:
    folder.mkdir(parents=True, exist_ok=True)

# Input tables
CT_AUDIT_CSV = OUT_AUDIT / "lung1_ct_series_audit.csv"
ALL_SERIES_CSV = OUT_AUDIT / "lung1_all_series_inventory.csv"
LUNG1_JOIN_CSV = OUT_PREML / "lung1_join_clinical_ct_OS2y.csv"

# PyRadiomics YAML
PARAMS_YAML = PROJECT_ROOT / "params_clean_updated.yaml"

# Outputs
OUT_FEATURES_CSV = OUT_RADIOMICS / "lung1_radiomics_features.csv"
OUT_QC_CSV = OUT_QC / "lung1_radiomics_qc.csv"

# Options
FORCE_RERUN = False
ROI_PREFERRED_NAME = "GTV-1"
MIN_MASK_VOXELS = 200
KEEP_DIAGNOSTICS = False

print("LUNG1_ROOT:", LUNG1_ROOT)
print("LUNG1_ROOT exists:", LUNG1_ROOT.exists())
print("Using OneDrive path:", "OneDrive" in str(LUNG1_ROOT))

print("\nInput files:")
print("CT_AUDIT_CSV:", CT_AUDIT_CSV, CT_AUDIT_CSV.exists())
print("ALL_SERIES_CSV:", ALL_SERIES_CSV, ALL_SERIES_CSV.exists())
print("LUNG1_JOIN_CSV:", LUNG1_JOIN_CSV, LUNG1_JOIN_CSV.exists())
print("PARAMS_YAML:", PARAMS_YAML, PARAMS_YAML.exists())

print("\nOutput files:")
print("OUT_FEATURES_CSV:", OUT_FEATURES_CSV)
print("OUT_QC_CSV:", OUT_QC_CSV)

LUNG1_ROOT: C:\Users\Ideanuodo\Desktop\THESIS\ML Dataset\manifest-1603198545583\NSCLC-Radiomics
LUNG1_ROOT exists: True
Using OneDrive path: False

Input files:
CT_AUDIT_CSV: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_ct_series_audit.csv True
ALL_SERIES_CSV: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\01_dicom_audit\lung1_all_series_inventory.csv True
LUNG1_JOIN_CSV: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\02_preML_tables\lung1_join_clinical_ct_OS2y.csv True
PARAMS_YAML: C:\Users\Ideanuodo\Desktop\Updated Code\params_clean_updated.yaml True

Output files:
OUT_FEATURES_CSV: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\03_radiomics_extraction\lung1_radiomics_features.csv
OUT_QC_CSV: C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\04_qc_logs\lung1_radiomics_qc.csv


In [49]:
# ==========================================
# Cell 2: Required file checks
# ==========================================

required_paths = [
    LUNG1_ROOT,
    CT_AUDIT_CSV,
    ALL_SERIES_CSV,
    LUNG1_JOIN_CSV,
    PARAMS_YAML,
]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f"Required path not found:\n{p}")

print("All required paths exist.")

All required paths exist.


In [50]:
# ==========================================
# Cell 3: Helper functions
# ==========================================

def format_seconds(seconds: float) -> str:
    """Convert seconds into readable h/m/s format."""
    seconds = int(max(seconds, 0))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"


def read_header(fp: Path):
    """Read DICOM header only, without pixel data."""
    try:
        return pydicom.dcmread(str(fp), stop_before_pixels=True, force=True)
    except (InvalidDicomError, Exception):
        return None


def get_series_file_names_fast(example_file: Path, series_uid: str):
    """
    Fast path:
    Use the directory of the example CT slice and ask SimpleITK/GDCM
    for files belonging to the series UID.
    """
    try:
        series_dir = example_file.parent
        names = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(
            str(series_dir),
            series_uid
        )

        if names and len(names) > 0:
            return names

    except Exception:
        pass

    return None


def get_series_file_names_fallback(patient_dir: Path, series_uid: str):
    """
    Slow fallback:
    Recursively scan patient folder and keep CT files matching the series UID.
    """
    keep = []

    for fp in patient_dir.rglob("*"):
        if not fp.is_file():
            continue

        ds = read_header(fp)

        if ds is None:
            continue

        if getattr(ds, "Modality", "") != "CT":
            continue

        if getattr(ds, "SeriesInstanceUID", "") != series_uid:
            continue

        keep.append(fp)

    if len(keep) == 0:
        return []

    def inst_num(p):
        ds = read_header(p)

        if ds is None:
            return 0

        v = getattr(ds, "InstanceNumber", 0)

        try:
            return int(v)
        except Exception:
            return 0

    keep = sorted(keep, key=inst_num)

    return [str(p) for p in keep]


def load_ct_volume(patient_id: str, ct_example_file: Path, series_uid: str):
    """
    Load CT volume for one patient using the selected CT SeriesInstanceUID.
    """
    patient_dir = LUNG1_ROOT / patient_id

    names = get_series_file_names_fast(ct_example_file, series_uid)

    if names is None:
        names = get_series_file_names_fallback(patient_dir, series_uid)

    if not names:
        raise RuntimeError("No CT DICOM files found for the CT series UID.")

    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(names)
    img = reader.Execute()

    return img, names, ct_example_file.parent


def choose_roi_name(rtstruct):
    """
    Choose the GTV ROI from RTSTRUCT.
    Priority:
    1. exact ROI_PREFERRED_NAME
    2. any ROI containing GTV
    3. any ROI containing TUMOR/TUMOUR
    """
    names = rtstruct.get_roi_names()

    if ROI_PREFERRED_NAME in names:
        return ROI_PREFERRED_NAME

    upper = {n.upper(): n for n in names}

    for key in upper:
        if "GTV" in key:
            return upper[key]

    for key in upper:
        if "TUMOR" in key or "TUMOUR" in key:
            return upper[key]

    return None


def mask_np_to_sitk(mask_np: np.ndarray, ref_img: sitk.Image):
    """
    Convert RTSTRUCT numpy mask into SimpleITK mask image.
    Handles common axis-order differences.
    """
    sx, sy, sz = ref_img.GetSize()
    expected_zyx = (sz, sy, sx)

    if mask_np.shape == expected_zyx:
        arr = mask_np

    elif mask_np.shape == (sx, sy, sz):
        arr = np.transpose(mask_np, (2, 1, 0))

    elif mask_np.shape == (sy, sx, sz):
        arr = np.transpose(mask_np, (2, 0, 1))

    else:
        raise ValueError(
            f"Mask shape {mask_np.shape} does not match CT size {ref_img.GetSize()}."
        )

    arr = (arr > 0).astype(np.uint8)

    mask_img = sitk.GetImageFromArray(arr)
    mask_img.CopyInformation(ref_img)

    return mask_img


def compute_mask_stats(mask_img: sitk.Image):
    """Compute mask voxel count and volume in mL."""
    arr = sitk.GetArrayFromImage(mask_img)

    vox = int(np.count_nonzero(arr))

    sp = mask_img.GetSpacing()
    voxel_vol_mm3 = float(sp[0] * sp[1] * sp[2])
    vol_ml = vox * voxel_vol_mm3 / 1000.0

    return vox, vol_ml

In [51]:
# ==========================================
# Cell 4: Cache guard and input loading
# ==========================================

if (not FORCE_RERUN) and OUT_FEATURES_CSV.exists() and OUT_QC_CSV.exists():
    print("Found cached Lung1 radiomics outputs.")
    print("Features:", OUT_FEATURES_CSV.resolve())
    print("QC:", OUT_QC_CSV.resolve())

    df_out = pd.read_csv(OUT_FEATURES_CSV)
    df_qc = pd.read_csv(OUT_QC_CSV)

    RUN_EXTRACTION = False

else:
    print("No complete cache found, or FORCE_RERUN=True.")
    print("Will run Lung1 extraction.")

    ct_audit = pd.read_csv(CT_AUDIT_CSV)
    all_series = pd.read_csv(ALL_SERIES_CSV)
    lung1_join = pd.read_csv(LUNG1_JOIN_CSV)

    # Optional: drop Plastimatch case if present
    if "Manufacturer" in lung1_join.columns:
        lung1_join = lung1_join[
            lung1_join["Manufacturer"].fillna("") != "Plastimatch"
        ].copy()

    print("ct_audit:", ct_audit.shape)
    print("all_series:", all_series.shape)
    print("lung1_join:", lung1_join.shape)

    RUN_EXTRACTION = True

No complete cache found, or FORCE_RERUN=True.
Will run Lung1 extraction.
ct_audit: (422, 22)
all_series: (1265, 7)
lung1_join: (419, 18)


In [52]:
# ==========================================
# Cell 5: Build CT and RTSTRUCT lookup tables
# ==========================================

if RUN_EXTRACTION:
    ct_lookup = (
        ct_audit
        .set_index("PatientID")[["ExampleFile", "SeriesInstanceUID"]]
        .to_dict(orient="index")
    )

    rt_lookup = (
        all_series[all_series["Modality"] == "RTSTRUCT"]
        .drop_duplicates(subset=["PatientID"])
        .set_index("PatientID")[["ExampleFile"]]
        .to_dict(orient="index")
    )

    patient_ids = lung1_join["PatientID"].astype(str).tolist()

    print("Patients for extraction:", len(patient_ids))
    print("CT lookup entries:", len(ct_lookup))
    print("RTSTRUCT lookup entries:", len(rt_lookup))

    missing_ct = [p for p in patient_ids if p not in ct_lookup]
    missing_rt = [p for p in patient_ids if p not in rt_lookup]

    print("Missing CT lookup:", len(missing_ct))
    print("Missing RTSTRUCT lookup:", len(missing_rt))

    if len(missing_ct) > 0:
        print("First missing CT:", missing_ct[:10])

    if len(missing_rt) > 0:
        print("First missing RTSTRUCT:", missing_rt[:10])
else:
    print("Extraction skipped because cached files were loaded.")

Patients for extraction: 419
CT lookup entries: 422
RTSTRUCT lookup entries: 422
Missing CT lookup: 0
Missing RTSTRUCT lookup: 0


In [53]:
# ==========================================
# Cell 6: Run Lung1 radiomics extraction
# ==========================================

if RUN_EXTRACTION:
    extractor = featureextractor.RadiomicsFeatureExtractor(str(PARAMS_YAML))

    features_rows = []
    qc_rows = []

    total_patients = len(patient_ids)
    start_time = perf_counter()

    success_count = 0
    fail_count = 0

    progress_bar = tqdm(
        patient_ids,
        total=total_patients,
        desc="Extracting Lung1 radiomics",
        unit="patient",
        dynamic_ncols=True
    )

    for i, pid in enumerate(progress_bar, start=1):

        qc = {
            "PatientID": pid,
            "status": "ok",
            "error": "",
            "roi_used": "",
            "mask_voxels": np.nan,
            "mask_volume_ml": np.nan,
            "elapsed_at_patient": "",
            "eta_at_patient": "",
        }

        try:
            if pid not in ct_lookup:
                raise RuntimeError("Missing CT audit entry for patient.")

            if pid not in rt_lookup:
                raise RuntimeError("Missing RTSTRUCT entry for patient.")

            ct_example = Path(ct_lookup[pid]["ExampleFile"])
            ct_series_uid = str(ct_lookup[pid]["SeriesInstanceUID"])
            rtstruct_file = Path(rt_lookup[pid]["ExampleFile"])

            # Load CT volume
            ct_img, ct_files, ct_dir = load_ct_volume(
                patient_id=pid,
                ct_example_file=ct_example,
                series_uid=ct_series_uid
            )

            # Build RTSTRUCT mask
            rtstruct = RTStructBuilder.create_from(
                dicom_series_path=str(ct_dir),
                rt_struct_path=str(rtstruct_file)
            )

            roi_name = choose_roi_name(rtstruct)

            if roi_name is None:
                raise RuntimeError("Could not find a suitable GTV ROI name in RTSTRUCT.")

            qc["roi_used"] = roi_name

            mask_np = rtstruct.get_roi_mask_by_name(roi_name)
            mask_img = mask_np_to_sitk(mask_np, ct_img)

            # Mask QC
            mask_vox, mask_vol_ml = compute_mask_stats(mask_img)

            qc["mask_voxels"] = mask_vox
            qc["mask_volume_ml"] = mask_vol_ml

            if mask_vox < MIN_MASK_VOXELS:
                raise RuntimeError(f"Mask too small: {mask_vox} voxels.")

            # PyRadiomics extraction
            result = extractor.execute(ct_img, mask_img)

            row = {
                "PatientID": pid,
                "roi_used": roi_name,
                "mask_voxels": mask_vox,
                "mask_volume_ml": mask_vol_ml,
            }

            for k, v in result.items():
                if (not KEEP_DIAGNOSTICS) and str(k).startswith("diagnostics_"):
                    continue

                row[str(k)] = v

            features_rows.append(row)

            success_count += 1

        except Exception as e:
            qc["status"] = "fail"
            qc["error"] = str(e)

            fail_count += 1

        elapsed = perf_counter() - start_time
        avg_time_per_patient = elapsed / i
        remaining_patients = total_patients - i
        eta = avg_time_per_patient * remaining_patients

        qc["elapsed_at_patient"] = format_seconds(elapsed)
        qc["eta_at_patient"] = format_seconds(eta)

        qc_rows.append(qc)

        progress_bar.set_postfix({
            "current": pid,
            "ok": success_count,
            "fail": fail_count,
            "elapsed": format_seconds(elapsed),
            "ETA": format_seconds(eta),
        })

    print("Extraction complete.")
else:
    print("Extraction skipped.")

Extracting Lung1 radiomics: 100%|██████████| 419/419 [55:33<00:00,  7.96s/patient, current=LUNG1-422, ok=413, fail=6, elapsed=55m 33s, ETA=0s]

Extraction complete.


In [54]:
# ==========================================
# Cell 7: Save Lung1 radiomics outputs
# ==========================================

if RUN_EXTRACTION:
    df_feat = pd.DataFrame(features_rows)
    df_qc = pd.DataFrame(qc_rows)

    # Merge extracted features with clinical and CT metadata
    df_out = lung1_join.merge(df_feat, on="PatientID", how="left")

    df_out.to_csv(OUT_FEATURES_CSV, index=False)
    df_qc.to_csv(OUT_QC_CSV, index=False)

    print("Saved features:")
    print(OUT_FEATURES_CSV.resolve())

    print("\nSaved QC:")
    print(OUT_QC_CSV.resolve())

else:
    print("Using cached outputs:")
    print(OUT_FEATURES_CSV.resolve())
    print(OUT_QC_CSV.resolve())

Saved features:
C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\03_radiomics_extraction\lung1_radiomics_features.csv

Saved QC:
C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\04_qc_logs\lung1_radiomics_qc.csv


In [57]:
# ==========================================
# Create ML-ready Lung1 radiomics table
# Keeps only patients with successful extraction
# ==========================================

LUNG1_ML_READY_CSV = OUT_RADIOMICS / "lung1_radiomics_features_ml_ready.csv"

df_ml_ready = df_out[df_out["roi_used"].notna()].copy()

print("Original Lung1 output table:", df_out.shape)
print("ML-ready extracted table:", df_ml_ready.shape)

print("\nOS_2y distribution in ML-ready extracted cohort:")
display(df_ml_ready["OS_2y"].value_counts(dropna=False))

print("\nManufacturer distribution in ML-ready extracted cohort:")
display(df_ml_ready["Manufacturer"].value_counts(dropna=False))

print("\nSIEMENS kernel distribution in ML-ready extracted cohort:")
display(
    df_ml_ready.loc[df_ml_ready["Manufacturer"] == "SIEMENS", "ConvolutionKernel"]
    .replace("", np.nan)
    .value_counts(dropna=False)
)

df_ml_ready.to_csv(LUNG1_ML_READY_CSV, index=False)

print("\nSaved ML-ready Lung1 radiomics table:")
print(LUNG1_ML_READY_CSV.resolve())

Original Lung1 output table: (419, 128)
ML-ready extracted table: (413, 128)

OS_2y distribution in ML-ready extracted cohort:


OS_2y
1.0    248
0.0    165
Name: count, dtype: int64


Manufacturer distribution in ML-ready extracted cohort:


Manufacturer
SIEMENS      318
CMS, Inc.     95
Name: count, dtype: int64


SIEMENS kernel distribution in ML-ready extracted cohort:


ConvolutionKernel
B19f    141
B30f     73
B31s     30
B31f     25
B18f     23
B41f     19
B30s      6
B19s      1
Name: count, dtype: int64


Saved ML-ready Lung1 radiomics table:
C:\Users\Ideanuodo\Desktop\Updated Code\outputs_lung1_clean\03_radiomics_extraction\lung1_radiomics_features_ml_ready.csv


In [55]:
# ==========================================
# Cell 8: Extraction summary
# ==========================================

print("Feature/output table:", df_out.shape)
print("QC table:", df_qc.shape)

print("\nQC status summary:")
display(df_qc["status"].value_counts(dropna=False))

if "roi_used" in df_out.columns:
    print(
        "\nRows with extracted radiomics:",
        int(df_out["roi_used"].notna().sum()),
        "/",
        df_out.shape[0]
    )

failed = df_qc[df_qc["status"] == "fail"][["PatientID", "error"]]

if len(failed) > 0:
    print("\nFailed patients, first 20:")
    display(failed.head(20))
else:
    print("\nNo failed patients.")

Feature/output table: (419, 128)
QC table: (419, 8)

QC status summary:


status
ok      413
fail      6
Name: count, dtype: int64


Rows with extracted radiomics: 413 / 419

Failed patients, first 20:


,PatientID,error
17,LUNG1-018,Resegmentation excluded too many voxels with l...
34,LUNG1-035,Loaded RTStruct references image(s) that are n...
96,LUNG1-098,Resegmentation excluded too many voxels with l...
153,LUNG1-156,Resegmentation excluded too many voxels with l...
278,LUNG1-281,Resegmentation excluded too many voxels with l...
400,LUNG1-403,Resegmentation excluded too many voxels with l...
